# Задание 1

In [121]:
import pandas as pd

pd.set_option('display.max_columns', 100)
data = pd.read_csv(
    "school_progress.csv",
    index_col="enrollment_id"
)
print(data.head(5))
print(data.shape)
print(data.dtypes)
print(data.isna().sum())

               student_id   student_name  grade class_letter    city  \
enrollment_id                                                          
10001                2001    Мария Белов      5            A  Москва   
10002                2001    Мария Белов      5            A  Москва   
10003                2001    Мария Белов      5            A  Москва   
10004                2001    Мария Белов      5            A  Москва   
10005                2002  Мария Макаров      9            A   Псков   

                       course        teacher        track     level  \
enrollment_id                                                         
10001                  Python    Е.В. Орлова  programming     basic   
10002                Биология  Т.К. Лебедева      science  advanced   
10003           Робототехника    И.А. Петров  engineering     basic   
10004                  Физика   М.И. Соколов      science     basic   
10005          Веб-разработка     Д.Р. Белов  programming     basic  

# Задание 2

In [122]:
scores = data["average_score"]
print(scores.mean())
print(scores.median())
print(scores.max())
print(scores.min())
print(scores.isnull().sum())
print(scores.nlargest(5))
print((scores>=80).sum())


74.28333333333333
73.69999999999999
97.6
35.0
22
enrollment_id
10075    97.6
10020    96.6
10193    96.3
10264    96.3
10074    96.2
Name: average_score, dtype: float64
96


# Задание 3

In [123]:
data["progress_calc"] = ((data["lessons_completed"] / data["total_lessons"]) * 100).round(2)

data["progress_num"] = data["progress"].str.replace("%", "", regex=False).astype(float)

data["diff"] = (data["progress_num"] - data["progress_calc"]).abs()

max_difference = data["diff"].max()
print(f"Максимальная разница: {max_difference}")


anomalies = data[data["diff"] > 1]
print("Строки с разницей больше 1%:")
print(anomalies)

Максимальная разница: 0.5
Строки с разницей больше 1%:
Empty DataFrame
Columns: [student_id, student_name, grade, class_letter, city, course, teacher, track, level, start_date, end_date, status, lessons_completed, total_lessons, progress, average_score, hours_spent, attempts, email, clubs, comment, progress_calc, progress_num, diff]
Index: []


# Задание 4

In [124]:
condition_grade = data["grade"].between(8, 11)
condition_course = data["course"].isin(["Python", "Анализ данных", "Робототехника"])
condition_status = data["status"].isin(["active", "completed"])
condition_average_score = data["average_score"].notna() & (data["average_score"] >= 75)

candidates = data[condition_grade & condition_course & condition_status & condition_average_score]

keep = ["student_name", "grade", "course", "status",
"average_score", "hours_spent"]

candidates = candidates[keep]

candidates = candidates.sort_values(
    by=["average_score", "hours_spent"],
    ascending=[False, False]
)

print(candidates.head(10))

                   student_name  grade         course     status  \
enrollment_id                                                      
10020             Софья Яковлев     11         Python     active   
10138          Виктория Фёдоров     11         Python     active   
10090            Алексей Крылов      8  Анализ данных     active   
10178              Артём Иванов     11         Python  completed   
10194              Кирилл Орлов     10  Анализ данных  completed   
10270            Виктория Орлов      8  Анализ данных     active   
10059              Никита Орлов     10  Анализ данных  completed   
10165           Валерия Новиков      9         Python  completed   
10141            Андрей Фёдоров     11         Python     active   
10225              Алина Павлов      8         Python  completed   

               average_score  hours_spent  
enrollment_id                              
10020                   96.6          9.8  
10138                   95.1         21.8  
10090  

# Задание 5

In [125]:
clean = data.copy()
clean["comment_clean"] = clean["comment"].fillna("нет комментариев")
clean["comment_clean"] = clean["comment_clean"].str.lower()
# clean["email_login"] = clean["email"].str.split("@").str[0]
# clean["email_domain"] = clean["email"].str.split("@").str[1]
clean[["email_login", "email_domain"]] = clean["email"].str.split("@", expand = True)


print(clean["comment_clean"].head())
print(clean["email_login"].head())
print(clean["email_domain"].head())

clean["has_robotics_club"] = clean["clubs"].isin(["Робототехника"])

print(clean.head(5))

enrollment_id
10001          иногда опаздывает со сдачей
10002                  отличный прогресс  
10003    нужно внимательнее читать условие
10004                     нет комментариев
10005                     нет комментариев
Name: comment_clean, dtype: str
enrollment_id
10001    student2001
10002    student2001
10003    student2001
10004    student2001
10005    student2002
Name: email_login, dtype: str
enrollment_id
10001    school.example
10002    school.example
10003    school.example
10004    school.example
10005    school.example
Name: email_domain, dtype: str
               student_id   student_name  grade class_letter    city  \
enrollment_id                                                          
10001                2001    Мария Белов      5            A  Москва   
10002                2001    Мария Белов      5            A  Москва   
10003                2001    Мария Белов      5            A  Москва   
10004                2001    Мария Белов      5            A  Мос

# Задание 6

In [126]:
print(clean["average_score"].isna().sum())
print(clean["clubs"].isna().sum())
print(clean["comment"].isna().sum())
print(clean.duplicated(
    subset=["student_id", "course"],
    keep=False).sum())
print(clean.duplicated(subset=["student_id", "course"], keep="first").sum())
print(len(data) - len(clean))


22
23
26
12
6
0


# Задание 7

In [127]:
course_report = data.groupby("course").agg(
record = ("student_id", "count"),
students = ("student_id", "nunique"),
mean_score=("average_score", "mean"),
mean_hours=("hours_spent", "mean"),
mean_attempts=("attempts", "mean")
).reset_index()

course_report = course_report.sort_values(by="mean_score", ascending=False)

print("Топ-3 курса по среднему баллу:")
print(course_report.head(3))

Топ-3 курса по среднему баллу:
            course  record  students  mean_score  mean_hours  mean_attempts
2  Английский язык      43        40   75.494872   23.746512       2.790698
0           Python      39        39   75.286486   23.679487       2.717949
6    Робототехника      35        35   74.648387   29.082857       2.914286


# Задание 8

In [128]:
level_report = data.groupby(["course", "level"]).agg(
records = ("student_id", "count"),
mean_score = ("average_score", "mean"),
mean_hours = ("hours_spent", "mean"),
).reset_index()
level_report_filtered = level_report[level_report["records"]>=5]
print(level_report_filtered.sort_values(by='mean_score', ascending=False).head(5))

             course         level  records  mean_score  mean_hours
5     Анализ данных  intermediate        6   81.780000   25.750000
10         Биология         basic        9   80.455556   42.633333
7   Английский язык         basic       16   78.938462   19.393750
14   Веб-разработка  intermediate       11   77.900000   15.627273
2            Python  intermediate       18   77.829412   27.883333


# Задание 9

In [129]:
# enrollments = enrollments 
students = data[["student_id", "student_name", "grade", "class_letter", "city"]].copy().drop_duplicates(subset=["student_id"])  
enrollments = data[[
    "student_id", "course", "level", "status", 
    "average_score", "hours_spent", "attempts"
]].copy()

enrollments_to_check = data[[
    "student_id", "course", "level", "status", 
    "average_score", "hours_spent", "attempts"
]].copy()
print(enrollments.head(10))
enrollments = enrollments.merge(students, how="left", on="student_id")

print(enrollments.shape)
print(enrollments_to_check.shape)


               student_id                  course     level     status  \
enrollment_id                                                            
10001                2001                  Python     basic     active   
10002                2001                Биология  advanced  completed   
10003                2001           Робототехника     basic     active   
10004                2001                  Физика     basic     active   
10005                2002          Веб-разработка     basic     active   
10006                2002           Робототехника  advanced  completed   
10007                2002         Английский язык  advanced  completed   
10008                2002  Олимпиадная математика     basic     active   
10009                2003                  Python     basic  completed   
10010                2003         Английский язык  advanced  completed   

               average_score  hours_spent  attempts  
enrollment_id                                        
100

# Задание 10

In [130]:
pivot_table =enrollments.pivot_table(index = "student_id", columns="course", values ="average_score", aggfunc="mean")

print(pivot_table.head(4))
cource_count = pivot_table.count(axis=1)
print(cource_count.head(4))
print("ученики с результатами по 3 и более курсам")
print(cource_count[cource_count >=3].head(4))

course      Python  Анализ данных  Английский язык  Биология  Веб-разработка  \
student_id                                                                     
2001          56.1            NaN              NaN      68.7             NaN   
2002           NaN            NaN             84.4       NaN            52.3   
2003          71.1            NaN             84.6       NaN             NaN   
2004           NaN            NaN              NaN       NaN             NaN   

course      Олимпиадная математика  Робототехника  Физика  
student_id                                                 
2001                           NaN           56.6    82.6  
2002                          51.8           70.3     NaN  
2003                           NaN           95.5     NaN  
2004                          70.5           83.6     NaN  
student_id
2001    4
2002    4
2003    3
2004    2
dtype: int64
ученики с результатами по 3 и более курсам
student_id
2001    4
2002    4
2003    3
2005    4
d

# Задание 11

In [ ]:
clean["start_date"] = pd.to_datetime(clean["start_date"])
clean["end_date"] = pd.to_datetime(clean["end_date"])


clean["start_month"] = clean["start_date"].dt.month
clean["start_weekday"] = clean["start_date"].dt.weekday 
clean["course_duration_days"] = (clean["end_date"] - clean["start_date"]).dt.days


mean_score_by_month = clean.groupby("start_month")["average_score"].mean().reset_index()
print("Средний балл по месяцам начала обучения:")
print(mean_score_by_month)

completed_courses = clean[clean["status"] == "completed"]
mean_duration = completed_courses["course_duration_days"].mean()
print(f"Средняя продолжительность завершённых курсов: {mean_duration:.1f} дн.")

1. Средний балл по месяцам начала обучения:
   start_month  average_score
0            1      75.941176
1            2      70.684000
2            3      77.255882
3            4      74.017143
4            5      70.900000
5            9      74.805882
6           10      73.956000
7           11      72.466667
8           12      75.441379

2. Средняя продолжительность завершённых курсов: 136.5 дн.


# Задане 12

In [ ]:
columns_to_keep = [
    "grade",
    "course",
    "level",
    "lessons_completed",
    "total_lessons",
    "hours_spent",
    "attempts",
    "start_month",
    "start_weekday",
    "average_score",
]

ml_data = data[columns_to_keep].copy()
ml_data = ml_data.dropna(subset=["average_score"])

print("Размерность датасета (shape)")
print(ml_data.shape)

print("Типы данных (dtypes)")
print(ml_data.dtypes)

print("Количество пропусков в столбцах")
print(ml_data.isna().sum())

print(f"\nПропусков в 'average_score': {ml_data['average_score'].isna().sum()}")

ml_data.to_csv("ml_data.csv", index=False)
print("Файл ml_data.csv успешно сохранен.")

print("Первые 5 строк ml_data")
print(ml_data.head(5))

=== Размерность датасета (shape) ===
(286, 10)

=== Типы данных (dtypes) ===
grade                  int64
course                   str
level                    str
lessons_completed      int64
total_lessons          int64
hours_spent          float64
attempts               int64
start_month            int32
start_weekday          int32
average_score        float64
dtype: object

=== Количество пропусков в столбцах ===
grade                 0
course                0
level                 0
lessons_completed     0
total_lessons         0
hours_spent           0
attempts              0
start_month           0
start_weekday         0
average_score        22
dtype: int64

Пропусков в 'average_score': 22
Файл ml_data.csv успешно сохранен.

=== Первые 5 строк ml_data ===
               grade          course     level  lessons_completed  \
enrollment_id                                                       
10001              5          Python     basic                 10   
10002             